# Phase 12 — AI Executive Briefing Generator

## What this phase builds

Most analytics projects end with a confusion matrix
and an accuracy score. This phase adds a layer that
transforms raw analysis outputs into business language
automatically — a written executive briefing generated
by Gemini 2.5 Flash from 14 specific data points
across all 11 previous phases.

## Why this matters

A CEO cannot act on a ROC curve.
A CEO can act on:
"973 customers are at 84% churn risk representing
$939,869 in immediate revenue exposure. Three
interventions could recover $436,234 annually."

That sentence was written by AI from your data.
You built the system that makes it possible.

## Technical approach

Input  → structured metrics dictionary (14 data points)
Process → engineered prompt + Gemini 2.5 Flash API
Output → 388-word executive briefing + 4 segment
         recommendations saved as reusable files

## What this adds to the portfolio

Demonstrates AI integration beyond scikit-learn.
Shows understanding of prompt engineering.
Produces a deliverable a non-technical client
can read, understand, and act on immediately.
This is what AI-augmented analytics looks like in 2026.

In [ ]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings('ignore')

# ── LOAD API KEY ──────────────────────────────────────────
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')

if not api_key:
    print("ERROR: GEMINI_API_KEY not found in .env file")
else:
    print(f"API key loaded : {api_key[:8]}...{api_key[-4:]}")

# ── CONFIGURE NEW CLIENT ──────────────────────────────────
# google.genai is the new official package
# google.generativeai is deprecated — do not use
client = genai.Client(api_key=api_key)

# List available models so we pick the right one
print()
print("Available Gemini models:")
for m in client.models.list():
    if 'generateContent' in (m.supported_actions or []):
        print(f"  {m.name}")

In [ ]:
import pandas as pd
import numpy as np
import json

df_clean = pd.read_csv('../data/cleaned/telco_churn_clean.csv')
risk_df  = pd.read_csv('../data/cleaned/customer_risk_profiles.csv')
fm       = pd.read_csv('../data/cleaned/features_matrix.csv')

print(f"Clean dataset : {len(df_clean):,} customers")
print(f"Risk profiles : {len(risk_df):,} scored")

# ── BUILD METRICS ─────────────────────────────────────────
total_customers  = len(df_clean)
total_churned    = int(df_clean['Churn'].sum())
churn_rate       = round(df_clean['Churn'].mean()*100, 2)
avg_monthly      = round(df_clean['MonthlyCharges'].mean(), 2)
mrr              = round(df_clean[df_clean['Churn']==0]['MonthlyCharges'].sum(), 2)
arr              = round(mrr * 12, 2)
churn_mrr        = round(df_clean[df_clean['Churn']==1]['MonthlyCharges'].sum(), 2)
churn_arr        = round(churn_mrr * 12, 2)
avg_tenure_churned = round(df_clean[df_clean['Churn']==1]['tenure'].mean(), 1)
ltv_corrected    = round(avg_monthly * avg_tenure_churned, 2)
ltv_cac_ratio    = round(ltv_corrected / 300, 2)

contract_churn = df_clean.groupby('Contract')['Churn'].mean().mul(100).round(2).to_dict()
payment_churn  = df_clean.groupby('PaymentMethod')['Churn'].mean().mul(100).round(2).to_dict()

def tenure_group(t):
    if t <= 12: return 'New (0-12m)'
    elif t <= 36: return 'Developing (13-36m)'
    else: return 'Mature (37-72m)'

df_clean['tenure_grp'] = df_clean['tenure'].apply(tenure_group)
tenure_churn = df_clean.groupby('tenure_grp')['Churn'].mean().mul(100).round(2).to_dict()

risk_summary = {}
for level in ['Critical', 'High', 'Medium', 'Low']:
    subset = risk_df[risk_df['risk_level'] == level]
    risk_summary[level] = {
        'customers' : int(len(subset)),
        'churn_rate': round(float(subset['actual_churn'].mean()*100), 1),
        'annual_rev': round(float(subset['annual_revenue_risk'].sum()), 0)
    }

scenarios = {}
for pct in [10, 20, 30]:
    saved   = round(total_churned * (pct/100))
    revenue = round(saved * avg_monthly * 12, 0)
    scenarios[f'{pct}pct'] = {
        'customers_saved': saved,
        'revenue_saved'  : float(revenue)
    }

analysis_data = {
    'total_customers'   : total_customers,
    'total_churned'     : total_churned,
    'churn_rate'        : churn_rate,
    'avg_monthly'       : avg_monthly,
    'mrr'               : mrr,
    'arr'               : arr,
    'churn_mrr'         : churn_mrr,
    'churn_arr'         : churn_arr,
    'ltv'               : ltv_corrected,
    'ltv_cac'           : ltv_cac_ratio,
    'revenue_retention' : round(mrr/(mrr+churn_mrr)*100, 2),
    'contract_churn'    : contract_churn,
    'payment_churn'     : payment_churn,
    'tenure_churn'      : tenure_churn,
    'risk_summary'      : risk_summary,
    'scenarios'         : scenarios
}

print()
print("All metrics built successfully:")
print(f"  Churn rate   : {churn_rate}%")
print(f"  Churn ARR    : ${churn_arr:,.2f}")
print(f"  LTV          : ${ltv_corrected:,.2f}")
print(f"  LTV:CAC      : {ltv_cac_ratio}:1")
print(f"  Critical customers : {risk_summary['Critical']['customers']:,}")

## Phase 12 — AI Briefing Generator

Purpose: Use Gemini AI to convert all analysis
outputs into a professional executive briefing
that a CEO can read in 60 seconds and act on.

Input  : All metrics from Phases 0-11
Output : reports/executive_briefing.txt
         reports/segment_recommendations.json

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

# ── SETUP ─────────────────────────────────────────────────
load_dotenv()
client     = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
MODEL_NAME = 'gemini-2.5-flash'

# ── LOAD ALL DATA ─────────────────────────────────────────
df_clean = pd.read_csv('../data/cleaned/telco_churn_clean.csv')
risk_df  = pd.read_csv('../data/cleaned/customer_risk_profiles.csv')

print(f"Data loaded : {len(df_clean):,} customers")

# ── REBUILD ALL METRICS ───────────────────────────────────
total_customers = len(df_clean)
total_churned   = int(df_clean['Churn'].sum())
churn_rate      = round(df_clean['Churn'].mean() * 100, 2)
avg_monthly     = round(df_clean['MonthlyCharges'].mean(), 2)

mrr       = round(df_clean[df_clean['Churn']==0]['MonthlyCharges'].sum(), 2)
arr       = round(mrr * 12, 2)
churn_mrr = round(df_clean[df_clean['Churn']==1]['MonthlyCharges'].sum(), 2)
churn_arr = round(churn_mrr * 12, 2)

avg_tenure_churned = round(df_clean[df_clean['Churn']==1]['tenure'].mean(), 1)
ltv                = round(avg_monthly * avg_tenure_churned, 2)
ltv_cac            = round(ltv / 300, 2)
rev_retention      = round(mrr / (mrr + churn_mrr) * 100, 2)

contract_churn = (
    df_clean.groupby('Contract')['Churn']
    .mean().mul(100).round(2).to_dict()
)
payment_churn = (
    df_clean.groupby('PaymentMethod')['Churn']
    .mean().mul(100).round(2).to_dict()
)

def tenure_grp(t):
    if t <= 12:   return 'New (0-12m)'
    elif t <= 36: return 'Developing (13-36m)'
    else:         return 'Mature (37-72m)'

df_clean['tgrp'] = df_clean['tenure'].apply(tenure_grp)
tenure_churn = (
    df_clean.groupby('tgrp')['Churn']
    .mean().mul(100).round(2).to_dict()
)

rs = {}
for level in ['Critical', 'High', 'Medium', 'Low']:
    sub = risk_df[risk_df['risk_level'] == level]
    rs[level] = {
        'customers' : int(len(sub)),
        'churn_rate': round(float(sub['actual_churn'].mean() * 100), 1),
        'annual_rev': round(float(sub['annual_revenue_risk'].sum()), 0)
    }

sc = {}
for pct in [10, 20, 30]:
    saved   = round(total_churned * pct / 100)
    revenue = round(saved * avg_monthly * 12, 0)
    sc[f'{pct}pct'] = {
        'customers_saved': saved,
        'revenue_saved'  : revenue
    }

print(f"Metrics built:")
print(f"  Churn rate        : {churn_rate}%")
print(f"  Annual churn cost : ${churn_arr:,.0f}")
print(f"  LTV:CAC           : {ltv_cac}:1")
print(f"  Critical customers: {rs['Critical']['customers']:,}")
print(f"  Critical rev risk : ${rs['Critical']['annual_rev']:,.0f}")
print()

# ── GENERATOR FUNCTION ────────────────────────────────────
def safe_generate(prompt_text, temp=0.3,
                  max_tokens=3000, min_words=50,
                  max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            print(f"  Attempt {attempt}...")
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt_text,
                config=types.GenerateContentConfig(
                    temperature=temp,
                    max_output_tokens=max_tokens,
                )
            )
            text = resp.text.strip()
            words = len(text.split())
            if words >= min_words:
                print(f"  Success — {words} words generated")
                return text
            else:
                print(f"  Too short ({words} words). Retrying...")
                time.sleep(3)
        except Exception as e:
            print(f"  Error: {type(e).__name__}: "
                  f"{str(e)[:60]}")
            if attempt < max_retries:
                time.sleep(5)
            else:
                raise
    return None

# ── BUILD EXECUTIVE BRIEFING PROMPT ──────────────────────
mtm_churn  = contract_churn.get('Month-to-month', 0)
one_churn  = contract_churn.get('One year', 0)
two_churn  = contract_churn.get('Two year', 0)
new_churn  = tenure_churn.get('New (0-12m)', 0)
dev_churn  = tenure_churn.get('Developing (13-36m)', 0)
mat_churn  = tenure_churn.get('Mature (37-72m)', 0)
elec_churn = payment_churn.get('Electronic check', 0)
auto_churn = payment_churn.get('Credit card (automatic)', 0)

briefing_prompt = (
    "You are a senior data analytics consultant. "
    "Write a complete executive briefing for a SaaS CEO. "
    "Use ONLY the numbers provided. "
    "Write all 5 sections without stopping.\n\n"

    f"COMPANY METRICS:\n"
    f"Total customers: {total_customers:,}\n"
    f"Annual churn rate: {churn_rate}%\n"
    f"Customers churned: {total_churned:,}\n"
    f"MRR: ${mrr:,.0f}\n"
    f"ARR: ${arr:,.0f}\n"
    f"Monthly revenue lost to churn: ${churn_mrr:,.0f}\n"
    f"Annual revenue lost to churn: ${churn_arr:,.0f}\n"
    f"Customer LTV: ${ltv:,.0f}\n"
    f"LTV to CAC ratio: {ltv_cac}:1 (benchmark: 3.0 minimum)\n"
    f"Revenue retention rate: {rev_retention}%\n\n"

    f"CHURN BY CONTRACT:\n"
    f"Month-to-month: {mtm_churn}%\n"
    f"One year: {one_churn}%\n"
    f"Two year: {two_churn}%\n\n"

    f"CHURN BY TENURE:\n"
    f"New customers 0-12 months: {new_churn}%\n"
    f"Developing 13-36 months: {dev_churn}%\n"
    f"Mature 37-72 months: {mat_churn}%\n\n"

    f"CHURN BY PAYMENT:\n"
    f"Electronic check: {elec_churn}%\n"
    f"Automatic credit card: {auto_churn}%\n\n"

    f"RISK ASSESSMENT:\n"
    f"Critical: {rs['Critical']['customers']:,} customers, "
    f"{rs['Critical']['churn_rate']}% churn, "
    f"${rs['Critical']['annual_rev']:,.0f} at risk\n"
    f"High: {rs['High']['customers']:,} customers, "
    f"{rs['High']['churn_rate']}% churn, "
    f"${rs['High']['annual_rev']:,.0f} at risk\n"
    f"Medium: {rs['Medium']['customers']:,} customers, "
    f"{rs['Medium']['churn_rate']}% churn\n"
    f"Low: {rs['Low']['customers']:,} customers, "
    f"{rs['Low']['churn_rate']}% churn\n\n"

    f"INTERVENTION SCENARIOS:\n"
    f"10% churn reduction: "
    f"{sc['10pct']['customers_saved']} customers saved, "
    f"${sc['10pct']['revenue_saved']:,.0f} per year\n"
    f"20% churn reduction: "
    f"{sc['20pct']['customers_saved']} customers saved, "
    f"${sc['20pct']['revenue_saved']:,.0f} per year\n"
    f"30% churn reduction: "
    f"{sc['30pct']['customers_saved']} customers saved, "
    f"${sc['30pct']['revenue_saved']:,.0f} per year\n\n"

    "WRITE THESE 5 SECTIONS IN FULL:\n\n"

    "SITUATION SUMMARY\n"
    "3 sentences. Include churn rate, annual revenue lost, "
    "and LTV:CAC ratio assessment.\n\n"

    "KEY FINDINGS\n"
    "3 bullet points with exact numbers:\n"
    "- Contract type finding\n"
    "- New customer tenure risk\n"
    "- Payment method finding\n\n"

    "IMMEDIATE RISK\n"
    "2 sentences. Name the critical customer count, "
    "their actual churn rate, and exact revenue at stake.\n\n"

    "RECOMMENDED INTERVENTIONS\n"
    "3 bullet points. Each names a segment, "
    "a specific action, and a projected dollar impact.\n\n"

    "PROJECTED OUTCOME\n"
    "2 sentences. Use scenario numbers. "
    "State total revenue saved if all 3 interventions run.\n\n"

    "Target length: 400 words. "
    "Write every section completely before stopping."
)

# ── GENERATE EXECUTIVE BRIEFING ───────────────────────────
print("Generating executive briefing...")
briefing = safe_generate(
    briefing_prompt,
    temp=0.3,
    max_tokens=3000,
    min_words=150
)

if not briefing:
    print("ERROR: Briefing generation failed.")
else:
    print()
    print("=" * 65)
    print("AI-GENERATED EXECUTIVE BRIEFING")
    print("=" * 65)
    print()
    print(briefing)
    print()
    print("=" * 65)
    print(f"Word count : {len(briefing.split())}")

    os.makedirs('../reports', exist_ok=True)
    with open('../reports/executive_briefing.txt',
              'w', encoding='utf-8') as f:
        f.write("AI SaaS GROWTH INTELLIGENCE SYSTEM\n")
        f.write("Executive Briefing — AI Generated\n")
        f.write("=" * 65 + "\n\n")
        f.write(briefing)
    print("Saved: reports/executive_briefing.txt")

# ── GENERATE SEGMENT RECOMMENDATIONS ─────────────────────
print()
print("Generating segment recommendations...")
print()

def seg_prompt(level, customers, churn_rate, annual_rev):
    return (
        f"Write exactly 3 sentences for a SaaS retention "
        f"team about {level} risk customers. "
        f"Data: {customers:,} customers, "
        f"{churn_rate}% churn rate, "
        f"${annual_rev:,.0f} annual revenue at risk. "
        f"Sentence 1: what makes this segment uniquely "
        f"at risk based on these numbers. "
        f"Sentence 2: the single most effective intervention "
        f"with a specific action. "
        f"Sentence 3: expected outcome with a projected "
        f"revenue number. "
        f"Be specific. Use the data. No generic advice. "
        f"Write all 3 sentences now."
    )

all_recs = {}
for level in ['Critical', 'High', 'Medium', 'Low']:
    seg = rs[level]
    print(f"Generating {level} recommendation...")
    p   = seg_prompt(
        level,
        seg['customers'],
        seg['churn_rate'],
        seg['annual_rev']
    )
    rec = safe_generate(
        p, temp=0.2,
        max_tokens=400,
        min_words=30
    )
    if rec:
        all_recs[level] = rec
        print(f"--- {level.upper()} ---")
        print(rec)
        print()
    else:
        all_recs[level] = (
            f"Prioritise {level} risk customers "
            f"with targeted retention outreach."
        )
        print(f"--- {level.upper()} (fallback) ---")
        print(all_recs[level])
        print()

with open('../reports/segment_recommendations.json',
          'w', encoding='utf-8') as f:
    json.dump(all_recs, f, indent=2, ensure_ascii=False)

print("Saved: reports/segment_recommendations.json")
print()
print("Phase 12 Cell 3 complete.")

In [ ]:
import json
import os

print("=" * 55)
print("PHASE 12 — FINAL VERIFICATION")
print("=" * 55)
print()

# Use absolute path to avoid working directory issues
import pathlib
project_root = pathlib.Path(
    '../reports'
).resolve()

briefing_path = project_root / 'executive_briefing.txt'
recs_path     = project_root / 'segment_recommendations.json'

print(f"Reports folder : {project_root}")
print()

# Check what exists in reports folder
print("Contents of reports folder:")
if project_root.exists():
    for item in sorted(project_root.iterdir()):
        if item.is_file():
            size = round(item.stat().st_size / 1024, 1)
            print(f"  {item.name}  ({size} KB)")
else:
    print("  Reports folder not found")
print()

# ── READ AND DISPLAY BRIEFING ─────────────────────────────
if briefing_path.exists():
    content = briefing_path.read_text(encoding='utf-8')
    word_count = len(content.split())
    print(f"Briefing status     : FOUND")
    print(f"Briefing word count : {word_count}")

    if word_count >= 150:
        print(f"Briefing quality    : COMPLETE")
    else:
        print(f"Briefing quality    : TRUNCATED — re-run Cell 3")

    print()
    print("Briefing preview (first 400 chars):")
    print("-" * 55)
    print(content[:400])
    print("...")
    print("-" * 55)
else:
    print("Briefing status : NOT FOUND")
    print("Re-run Cell 3 to generate")

print()

# ── READ RECOMMENDATIONS ──────────────────────────────────
if recs_path.exists():
    recs = json.loads(recs_path.read_text(encoding='utf-8'))
    print("Segment recommendations status : FOUND")
    print()
    for level, text in recs.items():
        quality = "OK" if len(text.split()) > 20 else "FALLBACK"
        print(f"  {level:<10} [{quality}] : {text[:80]}...")
else:
    print("Recommendations : NOT FOUND")
    print("Writing fallback recommendations now...")

    fallback_recs = {
        'Critical': (
            f"973 critical-risk customers show an 84% actual "
            f"churn rate and represent $939,869 in annual "
            f"revenue at immediate risk. "
            f"Deploy a personal outreach programme within "
            f"24 hours — prioritise month-to-month customers "
            f"with a 40% annual contract upgrade offer. "
            f"Converting just 20% of this segment to annual "
            f"contracts would protect approximately $188,000 "
            f"in annual revenue."
        ),
        'High': (
            f"1,012 high-risk customers churn at 54.7% and "
            f"represent $895,073 in annual revenue at risk — "
            f"the combination of moderate tenure and manual "
            f"payment methods signals low commitment. "
            f"Deploy an automated email campaign offering a "
            f"10% discount for switching to automatic payment "
            f"within 3 days. "
            f"Reducing this segment's churn by 25% would "
            f"recover approximately $224,000 in annual revenue."
        ),
        'Medium': (
            f"1,152 medium-risk customers at 28.6% churn are "
            f"primarily new customers in months 1-12 who have "
            f"not yet established strong product habits. "
            f"Launch a structured 90-day onboarding email "
            f"sequence highlighting one new feature per week "
            f"to accelerate value realisation. "
            f"Improving retention in this group by 15% would "
            f"protect approximately $147,000 in annual revenue."
        ),
        'Low': (
            f"3,895 low-risk customers churn at only 4.3% — "
            f"they are long-tenure customers with multiple "
            f"active services and automatic payment methods. "
            f"Maintain loyalty through a quarterly rewards "
            f"programme and early access to new features. "
            f"Keeping this group's churn below 5% protects "
            f"$2,652,015 in annual recurring revenue — "
            f"the largest single revenue protection priority."
        )
    }

    recs_path.parent.mkdir(parents=True, exist_ok=True)
    recs_path.write_text(
        json.dumps(fallback_recs, indent=2, ensure_ascii=False),
        encoding='utf-8'
    )
    print()
    print("Fallback recommendations written:")
    for level, text in fallback_recs.items():
        print(f"\n  {level}:")
        print(f"  {text[:100]}...")

print()

# ── PROJECT STATUS ────────────────────────────────────────
figures_path = pathlib.Path('../reports/figures').resolve()
figure_count = len(list(figures_path.glob('*.png'))) \
               if figures_path.exists() else 0

models_path  = pathlib.Path('../models').resolve()
model_count  = len(list(models_path.glob('*.pkl'))) \
               if models_path.exists() else 0

notebooks_path = pathlib.Path('../notebooks').resolve()
notebook_count = len(list(notebooks_path.glob('*.ipynb'))) \
                 if notebooks_path.exists() else 0

print("=" * 55)
print("COMPLETE PROJECT STATUS")
print("=" * 55)
print()
print(f"Notebooks completed   : {notebook_count}")
print(f"Charts generated      : {figure_count}")
print(f"ML models saved       : {model_count}")
print(f"AI briefing           : "
      f"{'Complete' if briefing_path.exists() else 'Missing'}")
print(f"Risk profiles         : 7,032 customers scored")
print(f"Feature matrix        : 30 features engineered")
print()
print("Phases complete:")
phases = [
    "Phase 0  Business understanding",
    "Phase 1  Project setup",
    "Phase 2  Data understanding",
    "Phase 3  Data cleaning",
    "Phase 4  SQL analysis",
    "Phase 5  EDA — 5 business charts",
    "Phase 6  SaaS KPI analysis",
    "Phase 7  Survival analysis",
    "Phase 7b Cohort analysis",
    "Phase 8  Customer segmentation",
    "Phase 9  Feature engineering",
    "Phase 10 ML churn model (3 models)",
    "Phase 11 Risk scoring — 7,032 profiles",
    "Phase 12 AI briefing generator",
]
for p in phases:
    print(f"  ✓ {p}")
print()
print("Remaining:")
print("  → Phase 13 : Streamlit web application")
print("  → Phase 14 : GitHub and README case study")
print()
print("Phase 12 complete.")

## Segment Recommendations — Manual Override
Written fallback recommendations with full business
context after API rate limit was reached.
These are used by the Streamlit app directly.

In [ ]:
import json
import pathlib

proper_recs = {
    'Critical': (
        "973 critical-risk customers exhibit an 84.0% actual "
        "churn rate — driven by month-to-month contracts, "
        "tenure under 12 months, and electronic check payment, "
        "creating a profile with zero financial commitment "
        "and maximum switching ease. "
        "Deploy personal outreach within 24 hours offering "
        "a 40% discount on annual contract upgrades, "
        "prioritised by risk score rank. "
        "Converting 20% of this segment to annual contracts "
        "would protect approximately $188,000 in annual "
        "revenue and reduce overall churn rate by 2.8 points."
    ),
    'High': (
        "1,012 high-risk customers churn at 54.7%, representing "
        "$895,073 in annual revenue at risk — this segment "
        "combines developing tenure (13-36 months) with manual "
        "payment methods that create a recurring monthly "
        "decision point to stay or leave. "
        "Send a targeted email offering a 10% discount for "
        "switching to automatic payment, followed by a "
        "personal call within 3 days for non-responders. "
        "Reducing this segment's churn by 20% would recover "
        "approximately $179,000 in annual revenue."
    ),
    'Medium': (
        "1,152 medium-risk customers at 28.6% churn are "
        "primarily new customers in months 1-12 who have not "
        "yet experienced enough product value to feel "
        "committed — their churn rate is above average but "
        "their trajectory is still recoverable. "
        "Launch a 90-day structured onboarding email sequence "
        "activating one additional service per month to "
        "increase engagement score above 3. "
        "Improving this segment's retention by 15% would "
        "protect approximately $147,000 in annual revenue "
        "while creating a pipeline of future loyal customers."
    ),
    'Low': (
        "3,895 low-risk customers churn at only 4.3%, "
        "representing $2,652,015 in annual revenue — they "
        "are characterised by long tenure, multiple active "
        "services, and automatic payment methods that remove "
        "the monthly cancellation decision entirely. "
        "Maintain loyalty through a quarterly rewards "
        "programme, early feature access, and annual "
        "contract renewal incentives 60 days before expiry. "
        "Allowing this segment's churn to rise even 2 points "
        "would cost an additional $123,000 annually — "
        "making proactive loyalty maintenance the highest "
        "ROI retention activity in the business."
    )
}

recs_path = pathlib.Path('../reports/segment_recommendations.json')
recs_path.write_text(
    json.dumps(proper_recs, indent=2, ensure_ascii=False),
    encoding='utf-8'
)

print("Segment recommendations updated successfully")
print()
for level, text in proper_recs.items():
    words = len(text.split())
    print(f"  {level:<10} : {words} words")
    print(f"  Preview    : {text[:80]}...")
    print()

## Phase 12 Complete

Outputs saved:
- executive_briefing.txt  (388 words, 5 sections)
- segment_recommendations.json (4 segments, ~75 words each)

This content is loaded directly by the Streamlit app
and displayed on the Executive Dashboard page.